# MLOps Tutorial: Handling Data Drift in Production

Welcome! This notebook will teach you about **MLOps** (Machine Learning Operations) - the practice of deploying and maintaining machine learning models in production.

## What You'll Learn

1. **Data Drift**: How data changes over time in production
2. **Model Retraining**: When and how to update models
3. **Deployment Strategies**: Safe ways to roll out new models
4. **Champion/Challenger Pattern**: Testing new models against production models
5. **Canary Releases**: Gradual rollouts to minimize risk

## The Scenario

Imagine you're managing a machine learning system that predicts customer behavior. Over time:

- Customer preferences change
- Market conditions shift
- Your model's accuracy degrades

**Your challenge**: Keep the system performing well despite these changes!

---

Let's get started! 🚀


## Part 1: Setup and Imports

In [ ]:
# @title First, we'll import the libraries we need and set up our environment. (run this!) {display-mode: "form"}
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def generate_batch(angle=0.0, size=400):
    """
    Generate a batch of binary classification data with drift.

    Drift is injected as a *rotation*: the entire dataset (both classes)
    is rotated around the origin by `angle` radians. As the angle grows
    over time, the whole data distribution rotates, simulating drift.

    Before rotation, class 0 is centered near (0,0) and class 1 near (2,2).

    Parameters:
    -----------
    angle : float
        The drift amount, expressed as a rotation angle (in radians),
        applied to the whole dataset.
    size : int
        Total number of samples (split equally between classes)

    Returns:
    --------
    X : np.ndarray
        Feature matrix of shape (size, 2)
    y : np.ndarray
        Labels of shape (size,)
    """
    half = size // 2
    # Class 0 (near origin)
    X0 = np.random.normal(loc=[0.0, 0.0], scale=[1.0, 1.0], size=(half, 2))
    y0 = np.zeros(half, dtype=int)
    # Class 1
    X1 = np.random.normal(loc=[2.0, 2.0], scale=[1.0, 1.0], size=(half, 2))
    y1 = np.ones(half, dtype=int)
    X = np.vstack([X0, X1])
    y = np.concatenate([y0, y1])

    # Inject drift by rotating the whole dataset around the origin
    rotation = np.array([
        [np.cos(angle), -np.sin(angle)],
        [np.sin(angle), np.cos(angle)],
    ])
    X = X @ rotation.T

    return X, y


def drift_schedule(round_idx, total_rounds, step_scale=0.05):
    """
    Calculate drift step using random walk that tends to increase over time.

    The returned step is added to the current rotation angle (in radians).

    Parameters:
    -----------
    round_idx : int
        Current round index
    total_rounds : int
        Total number of rounds in simulation
    step_scale : float
        Scale parameter for the random step variance

    Returns:
    --------
    float
        The drift step (rotation increment, in radians) to add to current angle
    """
    step = np.random.normal(loc=0.03, scale=step_scale * (1 + round_idx / total_rounds))
    return step


def plot_drift_over_time(drift_values, rounds):
    """
    Plot data drift (rotation angle) over rounds.

    Parameters:
    -----------
    drift_values : list
        List of drift values (rotation angles, in radians) for each round
    rounds : int
        Total number of rounds
    """
    plt.figure()
    plt.plot(range(1, rounds + 1), drift_values)
    plt.title("Data Drift (rotation angle) over Rounds")
    plt.xlabel("Round")
    plt.ylabel("Rotation angle (radians)")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_model_accuracies(acc_A_hist, acc_B_hist, retrain_rounds, promotions, rounds):
    """
    Plot model accuracies over time with retrain and promotion markers.

    Parameters:
    -----------
    acc_A_hist : list
        Accuracy history for Model A
    acc_B_hist : list
        Accuracy history for Model B
    retrain_rounds : list
        Rounds where Model B was retrained
    promotions : list
        Rounds where champion promotion occurred
    rounds : int
        Total number of rounds
    """
    plt.figure()
    plt.plot(range(1, rounds + 1), acc_A_hist, label="Model A (static)", linewidth=2)
    plt.plot(range(1, rounds + 1), acc_B_hist, label="Model B (retrained)", linewidth=2)

    # Mark retraining rounds
    for rr in retrain_rounds:
        plt.axvline(rr, linestyle=":", alpha=0.6, color="gray")

    # Mark promotion rounds
    for pr in promotions:
        plt.axvline(pr, linestyle="--", alpha=0.8, color="red", linewidth=1.5)

    plt.title("Accuracy over Rounds (vertical: retrain ':', promotion '--')")
    plt.xlabel("Round")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_served_accuracy(served_acc_hist, rounds):
    """
    Plot blended (served) accuracy over rounds.

    This shows the actual accuracy experienced by users
    when using canary routing.

    Parameters:
    -----------
    served_acc_hist : list
        History of blended accuracy values
    rounds : int
        Total number of rounds
    """
    plt.figure()
    plt.plot(range(1, rounds + 1), served_acc_hist, linewidth=2, color="green")
    plt.title("Served (Blended) Accuracy over Rounds")
    plt.xlabel("Round")
    plt.ylabel("Served Accuracy")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_sample_data(X_sample, y_sample):
    """
    Visualize a sample of the binary classification data.

    Shows the two classes in a 2D scatter plot to help understand
    the data distribution before drift occurs.

    Parameters:
    -----------
    X_sample : np.ndarray
        Feature matrix of shape (n_samples, 2)
    y_sample : np.ndarray
        Labels of shape (n_samples,)
    """
    plt.figure(figsize=(8, 6))
    plt.scatter(X_sample[y_sample == 0, 0], X_sample[y_sample == 0, 1], alpha=0.5, label="Class 0 (stationary)", s=30)
    plt.scatter(X_sample[y_sample == 1, 0], X_sample[y_sample == 1, 1], alpha=0.5, label="Class 1 (will drift)", s=30)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title("Sample Data (No Drift)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Set random seed for reproducibility
np.random.seed(42)
plt.rcParams["figure.figsize"] = (10, 5)

print("✓ Setup complete!")

## Part 2: Configuration Parameters

These parameters control how our simulation runs. You can experiment with these later!

### What do these mean?

**Simulation Settings:**

- `ROUNDS`: How many time periods we'll simulate (like days or weeks)
- `BATCH_SIZE`: How many predictions we make each round

**Model B Retraining:**

- `RETRAINING_FREQUENCY`: How often we retrain Model B (every N rounds)
- `FRESHNESS_RELEVANCE`: How much we prioritize recent data (0.0 = treat all data equally, 1.0 = only focus on new data)

**Champion/Challenger:**

- `PROMOTION_THRESHOLD`: How much better a model must be to get promoted (2% accuracy)
- `PROMOTION_PATIENCE`: How many rounds in a row it must be better

**Canary Release:**

- `CANARY_FRACTION`: What % of traffic goes to the challenger (20%)
- `CANARY_PROMOTION_THRESHOLD`: Performance threshold for canary promotion
- `CANARY_MIN_ROUNDS`: Consecutive successful canary rounds needed


In [ ]:
# --- Simulation parameters ---
ROUNDS = 50  # number of deployment rounds
BATCH_SIZE = 400  # number of samples per round

# --- Model B (retraining) controls ---
RETRAINING_FREQUENCY = 5  # how often to retrain (in rounds)
FRESHNESS_RELEVANCE = 0.8  # 0.0 = keep old data dominant, 1.0 = prioritize newest data heavily

# --- Promotion controls ---
PROMOTION_THRESHOLD = 0.02  # B must beat A by this absolute accuracy margin
PROMOTION_PATIENCE = 3  # number of consecutive rounds meeting threshold to promote

# --- Canary controls ---
CANARY_FRACTION = 0.25  # share of traffic served by challenger each round

print(f"✓ Configuration set: {ROUNDS} rounds, retraining every {RETRAINING_FREQUENCY} rounds")

## Part 3: Understanding the Data

Our dataset is a **binary classification problem** - we're predicting one of two outcomes (0 or 1).

Think of it like:

- Will a customer buy? (Yes/No)
- Is this transaction fraud? (Yes/No)
- Will a user click the ad? (Yes/No)

The data has **two classes**:

- **Class 0**: Stays in roughly the same place (stationary)
- **Class 1**: **Drifts over time** - this simulates how real-world data changes!

In [ ]:
# @title Let's generate and visualize a sample batch: {display-mode: "form"}
# Generate a sample batch with no drift
X_sample, y_sample = generate_batch(angle=0.0, size=BATCH_SIZE)

print(f"Sample shape: {X_sample.shape}")
print(f"Features (X): {X_sample.shape[1]} dimensions")
print(f"Labels (y): {len(y_sample)} samples")
print(f"Class distribution: {np.sum(y_sample == 0)} class 0, {np.sum(y_sample == 1)} class 1")

# Visualize the data using our utility function
plot_sample_data(X_sample, y_sample)

## Part 4: Initial Training

Now we'll train our initial models. We need to:

1. **Generate initial training data** (2000 samples, no drift)
2. **Train Model A** (the Champion) - a Logistic Regression model
3. **Create Model B** (the Challenger) - initially a copy of Model A
4. **Evaluate both models** on a holdout set and report baseline accuracy


In [ ]:
# Step 1: Generate initial training data (no drift)
X_init, y_init = generate_batch(angle=0.0, size=2000)

print(f"✓ Generated {len(y_init)} training samples")

In [ ]:
# Step 2: Train Model A (Champion)
model_A = LogisticRegression(max_iter=1000)
model_A.fit(X_init, y_init)

print("✓ Model A trained!")

In [ ]:
# Step 3: Train Model B (Challenger) - YOUR CODE HERE! 🎯
# Model B should start with the same training as Model A

# HINT: Create a LogisticRegression model (same as Model A)
# HINT: Use max_iter=1000 as the parameter
# HINT: Fit it on X_init and y_init (same data as Model A)

model_B = ...  # YOUR CODE HERE
# YOUR CODE HERE to fit the model
# --- 🎯🎯🎯🎯 ---

print("✓ Model B trained (same initial data as Model A)")


In [ ]:
# @title Initialize training history {display-mode: "form"}
# Initialize Model B's training history
# We'll keep track of all the data B has seen for future retraining

X_hist = X_init.copy()
y_hist = y_init.copy()
w_hist = np.ones_like(y_hist, dtype=float)  # sample weights (all equal initially)

print(f"✓ Training history initialized with {len(y_hist)} samples")

In [ ]:
# Step 4: Evaluate both models on a holdout set
X_hold, y_hold = generate_batch(angle=0.0, size=1000)

# Compute baseline accuracy for Model A - PROVIDED FOR YOU
base_acc_A = accuracy_score(y_hold, model_A.predict(X_hold))

# Compute baseline accuracy for Model B - YOUR CODE HERE!
# HINT: Follow the same pattern as Model A above
# HINT: Use accuracy_score(y_true, y_pred)
# HINT: Get predictions using model_B.predict(X_hold)

base_acc_B = ...  # YOUR CODE HERE
# --- 🎯🎯🎯🎯 ---

print(f"\n{'=' * 60}")
print(f"BASELINE PERFORMANCE (No Drift)")
print(f"{'=' * 60}")
print(f"Model A (Champion): {base_acc_A:.3f}")
print(f"Model B (Challenger): {base_acc_B:.3f}")
print(f"{'=' * 60}\n")

### ✅ Check Your Work

Before proceeding, verify:

- Both models should have the **same** baseline accuracy (they're identical right now)
- Accuracy should be somewhere between 0.80 and 0.95 (this is a relatively easy problem)

If something looks wrong, go back and check your code!


## Part 5: Going Live - Deployment Simulation

Now we'll simulate what happens in production over 50 rounds. Each round represents a time period (day, week, etc.).

### What happens each round?

1. **Data drifts** - The whole dataset is rotated to a new orientation
2. **New data arrives** - We get a fresh batch of predictions to make
3. **Both models make predictions** - We evaluate how well they do
4. **Canary routing** - We blend predictions based on who's champion
5. **Check for promotion** - Should we switch champions?
6. **Retrain Model B** - If it's time, update Model B with new data

In [ ]:
# @title Let's set up the tracking variables first: {display-mode: "form"}
# Initialize tracking lists
drift_values = []  # Track how much drift has occurred
acc_A_hist = []  # Model A's accuracy each round
acc_B_hist = []  # Model B's accuracy each round
served_acc_hist = []  # Blended accuracy (what users experience)
retrain_rounds = []  # Which rounds we retrained Model B
promotions = []  # Which rounds we promoted Model B

# Model A is always the champion (static model)
# Model B is always the challenger (retrained model being tested)
promoted = False  # Track if B has been promoted to replace A
consecutive_wins = 0  # How many consecutive rounds B has beaten A

mu = 0.0  # Starting drift (no drift yet)

print("✓ Tracking variables initialized")
print(f"Champion: Model A (static)")
print(f"Challenger: Model B (retrained)")

## Part 6: The Main Simulation Loop

This is the heart of the simulation. We implement **realistic canary routing** where traffic is actually split between models.

### The Setup:

- **Model A (Champion)**: Static model, never retrained, serves most traffic (80%)
- **Model B (Challenger)**: Retrained model, serves canary traffic (20%), being tested for promotion

### What happens each round:

1. **Generate live batch** with current drift
2. **Split the batch realistically**:
   - 80% of requests → Model A (champion)
   - 20% of requests → Model B (challenger/canary)
3. **Each model serves its portion** - just like in production!
4. **Calculate served accuracy** - the weighted blend of what users experience
5. **Evaluate full performance** - both models on entire batch for promotion decisions
6. **Check promotion criteria** - should B replace A permanently?
7. **Retrain Model B** on schedule with freshness weighting

### Why This Approach?

In real production systems:

- You have a **stable champion** (Model A) that you trust
- You test a **new challenger** (Model B) on a small fraction of traffic
- If the challenger consistently beats the champion, you promote it
- After promotion, the new model serves all traffic (no more canary needed)


In [ ]:
# Main simulation loop
for r in range(1, ROUNDS + 1):
    # 1) Update drift (accumulated rotation angle, in radians)
    mu += drift_schedule(r, ROUNDS)
    drift_values.append(mu)

    # 2) Generate live batch with current drift (whole data rotated by mu)
    X_live, y_live = generate_batch(angle=mu, size=BATCH_SIZE)

    # 3) Split batch for realistic canary routing
    canary_size = int(BATCH_SIZE * CANARY_FRACTION)
    X_canary = X_live[:canary_size]  # Canary traffic (20%) → Model B (challenger)
    y_canary = y_live[:canary_size]
    X_champion = X_live[canary_size:]  # Champion traffic (80%) → Model A (champion)
    y_champion = y_live[canary_size:]

    # 4) Each model serves its assigned traffic
    # Model A (champion) serves most traffic (80%) - PROVIDED FOR YOU
    y_pred_champion = model_A.predict(X_champion)
    acc_champion = accuracy_score(y_champion, y_pred_champion)

    # Model B (challenger) serves canary traffic (20%) - YOUR CODE HERE! 🎯
    # HINT: Use model_B.predict(X_canary) to get predictions
    # HINT: Use accuracy_score(y_canary, y_pred_canary) to calculate accuracy

    y_pred_canary = ...  # YOUR CODE HERE
    acc_canary = ...  # YOUR CODE HERE
    # --- 🎯🎯🎯🎯 ---

    # 5) For tracking full model performance (on entire batch)
    # Model A's full performance - PROVIDED FOR YOU
    aA = accuracy_score(y_live, model_A.predict(X_live))

    # Model B's full performance - YOUR CODE HERE! 🎯
    # HINT: Follow the same pattern as Model A above
    aB = ...  # YOUR CODE HERE
    # --- 🎯🎯🎯🎯 ---

    # Store full accuracies for plotting
    acc_A_hist.append(aA)
    # YOUR CODE HERE: Append aB to acc_B_hist
    # --- 🎯🎯🎯🎯 ---

    # 6) Calculate served accuracy (what users actually experience)
    # Routing honors promotion: once Model B is promoted it serves ALL traffic;
    # until then it only serves the canary fraction. Without this, a promoted
    # challenger would never actually receive user traffic.
    effective_canary = 1.0 if promoted else CANARY_FRACTION
    served_acc = (1 - effective_canary) * acc_champion + effective_canary * acc_canary
    served_acc_hist.append(served_acc)

    # 7) Promotion logic: Check if B (challenger) should replace A (champion)
    # YOUR CODE HERE! 🎯
    if not promoted:  # Only check for promotion if B hasn't been promoted yet
        # TASK: Implement the promotion logic
        # HINT: Check if (aB - aA) >= PROMOTION_THRESHOLD
        # HINT: If yes, increment consecutive_wins
        # HINT: If no, reset consecutive_wins to 0
        # HINT: If consecutive_wins >= PROMOTION_PATIENCE, set promoted = True
        # HINT: When promoting, also append r to promotions and print a message

        pass  # YOUR CODE HERE
        # --- 🎯🎯🎯🎯 ---

    # 8) Retrain Model B on schedule - PROVIDED FOR YOU
    if r % RETRAINING_FREQUENCY == 0:
        # Decay old sample weights
        w_hist *= 1.0 - FRESHNESS_RELEVANCE

        # Add new data with fresh weights
        X_hist = np.vstack([X_hist, X_live])
        y_hist = np.concatenate([y_hist, y_live])
        w_new = np.full(shape=y_live.shape, fill_value=FRESHNESS_RELEVANCE, dtype=float)
        w_hist = np.concatenate([w_hist, w_new])

        # Retrain Model B with weighted data
        model_B.fit(X_hist, y_hist, sample_weight=w_hist)
        retrain_rounds.append(r)
        print(f"🔄 Round {r}: Model B retrained (total data: {len(y_hist)} samples)")

print(f"\n{'=' * 60}")
print(f"SIMULATION COMPLETE")
print(f"{'=' * 60}")
print(f"Model B promoted: {'Yes' if promoted else 'No'}")
if promoted and len(promotions) > 0:
    print(f"Promoted at round: {promotions[0]}")
print(f"Total retraining events: {len(retrain_rounds)}")
print(f"{'=' * 60}\n")

## Part 7: Visualization and Analysis

Now let's visualize what happened during the simulation!

### Plot 1: Data Drift Over Time

In [ ]:
# @title Plot: This shows how much the data distribution has shifted from the original. {display-mode: "form"}
plot_drift_over_time(drift_values, ROUNDS)

### Plot 2: Model Accuracies Over Time

This shows how each model performed:

- **Dotted vertical lines (`:`)**: When Model B was retrained
- **Dashed vertical lines (`--`)**: When champion was promoted

**Watch for:**

- Does Model A's accuracy decline as drift increases?
- Does Model B maintain better accuracy after retraining?
- Do promotions happen when Model B is clearly better?


In [ ]:
# @title Plot: Model Accuracies {display-mode: "form"}
plot_model_accuracies(acc_A_hist, acc_B_hist, retrain_rounds, promotions, ROUNDS)

### Plot 3: Served Accuracy (What Users Experience)

This shows the **blended accuracy** from canary routing - what your users actually experience.

Notice it's often **smoother** than individual model performance because we're blending predictions.


In [ ]:
# @title Plot: Served Accuracy {display-mode: "form"}
plot_served_accuracy(served_acc_hist, ROUNDS)

## Part 8: Summary Statistics

In [ ]:
# @title Let's calculate some key metrics to understand performance: {display-mode: "form"}
# Calculate average accuracies
avg_acc_A = np.mean(acc_A_hist)
avg_acc_B = np.mean(acc_B_hist)
avg_served = np.mean(served_acc_hist)

# Calculate final accuracies (last 10 rounds)
final_acc_A = np.mean(acc_A_hist[-10:])
final_acc_B = np.mean(acc_B_hist[-10:])
final_served = np.mean(served_acc_hist[-10:])

# Calculate accuracy degradation for Model A
initial_acc_A = np.mean(acc_A_hist[:5])
degradation = initial_acc_A - final_acc_A

print(f"\n{'=' * 60}")
print(f"PERFORMANCE SUMMARY")
print(f"{'=' * 60}")
print(f"\nAverage Accuracy (all rounds):")
print(f"  Model A (static):     {avg_acc_A:.3f}")
print(f"  Model B (retrained):  {avg_acc_B:.3f}")
print(f"  Served (blended):     {avg_served:.3f}")
print(f"\nFinal Accuracy (last 10 rounds):")
print(f"  Model A (static):     {final_acc_A:.3f}")
print(f"  Model B (retrained):  {final_acc_B:.3f}")
print(f"  Served (blended):     {final_served:.3f}")
print(f"\nModel A Degradation:")
print(f"  Initial: {initial_acc_A:.3f} → Final: {final_acc_A:.3f}")
print(f"  Loss: {degradation:.3f} ({degradation * 100:.1f}% points)")
print(f"\nMLOps Events:")
print(f"  Retraining events:  {len(retrain_rounds)}")
print(f"  Model B promoted:   {'Yes' if promoted else 'No'}")
if promoted and len(promotions) > 0:
    print(f"  Promoted at round:  {promotions[0]}")
print(f"{'=' * 60}\n")

## Part 9: Reflection Questions 🤔

Take a moment to think about what you've learned:

### Questions to Consider:

1. **Why did Model A's accuracy decline?**

2. **How did retraining help Model B?**

3. **Why use canary releases instead of immediately switching?**

4. **What if we retrained more frequently?**

5. **What does FRESHNESS_RELEVANCE control?**

---


## Part 10: Experiments - Try It Yourself! 🧪

Now it's time to experiment! Go back to **Part 2** and try changing these parameters:

### Experiment Ideas:

1. **More Frequent Retraining**
   - Change `RETRAINING_FREQUENCY` from 5 to 2
   - What happens to Model B's performance?
   - What's the trade-off? (More retraining = more computational cost!)

2. **Higher Freshness Relevance**
   - Change `FRESHNESS_RELEVANCE` from 0.5 to 0.8
   - Does Model B adapt faster to drift?
   - Does it become less stable?

3. **Stricter Promotion Criteria**
   - Change `PROMOTION_THRESHOLD` from 0.02 to 0.05
   - Change `PROMOTION_PATIENCE` from 3 to 5
   - How does this affect champion switches?

4. **Larger Canary**
   - Change `CANARY_FRACTION` from 0.20 to 0.50
   - How does this affect the served accuracy?
   - What's the risk/benefit trade-off?

5. **No Retraining**
   - Set `RETRAINING_FREQUENCY` to 999 (effectively never)
   - What happens? Both models should perform similarly (both static)

### Record Your Findings:

Use the cell below to document what you discover!


### My Experiment Results:

_(Double-click to edit)_

**Experiment 1:**

- Parameters changed:
- Observation:
- Learning:

**Experiment 2:**

- Parameters changed:
- Observation:
- Learning:

**Experiment 3:**

- Parameters changed:
- Observation:
- Learning:


## Part 11: Key Takeaways 🎓

Congratulations! You've completed the MLOps tutorial. Here's what you learned:

### Core MLOps Concepts:

1. **Data Drift**
   - Real-world data changes over time
   - Static models degrade in performance
   - Monitoring is essential!

2. **Model Retraining**
   - Regular retraining helps models adapt
   - Frequency is a trade-off: adaptation vs. cost
   - Freshness weighting prioritizes recent patterns

3. **Champion/Challenger Pattern**
   - Keep a stable champion in production
   - Test challengers before full deployment
   - Use evidence-based promotion criteria

4. **Canary Releases**
   - Gradual rollout minimizes risk
   - Blend predictions to maintain stability
   - Monitor performance before full promotion

5. **Safety Mechanisms**
   - Promotion thresholds prevent noise-driven changes
   - Patience requirements avoid "flapping"
   - Blended serving protects user experience

### Real-World Applications:

These techniques are used by companies like:

- **Netflix**: Recommendation system updates
- **Uber**: Demand prediction models
- **Amazon**: Fraud detection systems
- **Google**: Ad ranking algorithms

### Next Steps:

To deepen your MLOps knowledge:

1. Learn about **monitoring and alerting** systems
2. Study **A/B testing** methodologies
3. Explore **feature stores** for managing data
4. Investigate **model versioning** and **experiment tracking**
5. Read about **ML pipelines** and **automation**

---

**Well done!** 🎉 You now understand the fundamentals of keeping ML systems healthy in production!
